In [1]:
!pip3 install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.7 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import cv2 as cv
from google.colab.patches import cv2_imshow
from tqdm import tqdm

In [4]:
# == Download pretrained model ==

%cd /content/ByteTrack/
%mkdir pretrained
%cd pretrained

# == Download pretrained X model weights ==
!gdown --id "1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5"

[Errno 2] No such file or directory: '/content/ByteTrack/'
/content
/content/pretrained
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5
From (redirected): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5&confirm=t&uuid=2617ad14-fda0-4754-ad33-69bdf6d540bb
To: /content/pretrained/bytetrack_x_mot17.pth.tar
100% 793M/793M [00:16<00:00, 48.3MB/s]


In [5]:
# == Install dependencies ==
!pip3 install cython
!pip3 install 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
!pip3 install cython_bbox

%cd /content/ByteTrack/
!pip3 install -r requirements.txt

  Cloning https://github.com/cocodataset/cocoapi.git to /tmp/pip-req-build-piay8xqc
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/cocoapi.git /tmp/pip-req-build-piay8xqc
  Resolved https://github.com/cocodataset/cocoapi.git to commit 8c9bcc3cf640524c4c20a9c40e89cb6a2f2fa0e9
  Preparing metadata (setup.py) ... done
  Created wheel for pycocotools: filename=pycocotools-2.0-cp312-cp312-linux_x86_64.whl size=426683 sha256=480333673831af871011647c4f4715a58ae762267667bfc039f68bd9ff9c9857
  Stored in directory: /tmp/pip-ephem-wheel-cache-qt4aakc8/wheels/95/e6/c7/8ceda667bca7218619fea052622a0b11a37fb51c28c993fae3
Successfully built pycocotools
  Attempting uninstall: pycocotools
    Found existing installation: pycocotools 2.0.10
    Uninstalling pycocotools-2.0.10:
      Successfully uninstalled pycocotools-2.0.10
  Preparing metadata (setup.py) ... done
  Created wheel for cython_bbox: filename=cython_bbox-0.1.5-cp312-cp312-linux_x86_64.whl size=111559

In [6]:
# == Install ByteTrack ==
!python3 setup.py install develop

python3: can't open file '/content/pretrained/setup.py': [Errno 2] No such file or directory


In [7]:
!pip3 install loguru sap lap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 6.8 MB/s eta 0:00:00


In [10]:
#Load a YOLO model

model = YOLO('yolov5s.pt')

#Load the video capture

videoCap = cv.VideoCapture('/content/YTDown.com_Shorts_When-Saka-was-playing-LEFT-BACK-for-Arse_Media_nXnl4cAGXJU_002_720p (1).mp4')
weight = int(videoCap.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(videoCap.get(cv.CAP_PROP_FRAME_HEIGHT))
frames_per_second = int(videoCap.get(cv.CAP_PROP_FPS)) or 30
num_frames = int(videoCap.get(cv.CAP_PROP_FRAME_COUNT))
fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter('output.mp4', fourcc, frames_per_second, (weight, height))

PRO TIP 💡 Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



In [11]:
#get class colors

def classCol(cls_num):
  base_col = (255,0,0), (0,255,0), (0,0,255)
  col_idx = cls_num % len(base_col)
  increment = [(1, -2, 1), (-2, 1, -1), (1, -1, 2)]
  color = [base_col[col_idx][i] + increment[col_idx][i] * (cls_num // len(base_col)) % 256 for i in range(3)]
  return tuple(color)

# while True:
for frame_i in tqdm(range(num_frames+2)):

    ret, frame = videoCap.read()
    if not ret:
        break
    results = model.track(frame, tracker='bytetrack.yaml')

    for result in results:
        # get the classes names and object id
        classes_names = result.names

        boxes = results[0].boxes.xyxy.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        box_classes = results[0].boxes.cls.int().cpu().tolist()


        # iterate over each box
        for box, tid, cls in zip(boxes, track_ids, box_classes):

            # get coordinates
            [x1, y1, x2, y2] = box
            # convert to int
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            # get the class
            cls = int(cls)

            # get the class name
            class_name = classes_names[cls]

            # get the respective colour
            colour = classCol(cls)

            # draw the rectangle
            cv.rectangle(frame, (x1, y1), (x2, y2), colour, 2)

            # put the class name and confidence on the image
            cv.putText(frame, f'{classes_names[cls]} {tid}', (x1, y1), cv.FONT_HERSHEY_SIMPLEX, 1, colour, 2)

    # show the image
    # cv2_imshow('frame', frame)
    out.write(frame)

    # break the loop if 'q' is pressed
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

# release the video capture and destroy all windows
videoCap.release()
out.release()
cv.destroyAllWindows()


  0%|          | 0/1489 [00:00<?, ?it/s]


0: 640x384 5 persons, 2 sports balls, 67.1ms
Speed: 21.2ms preprocess, 67.1ms inference, 417.5ms postprocess per image at shape (1, 3, 640, 384)


  0%|          | 1/1489 [00:02<58:09,  2.34s/it]


0: 640x384 6 persons, 2 sports balls, 10.2ms
Speed: 3.9ms preprocess, 10.2ms inference, 5.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 5 persons, 1 sports ball, 10.1ms
Speed: 2.2ms preprocess, 10.1ms inference, 5.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 6 persons, 10.1ms
Speed: 3.8ms preprocess, 10.1ms inference, 20.4ms postprocess per image at shape (1, 3, 640, 384)


  0%|          | 4/1489 [00:02<12:16,  2.02it/s]

...

100%|█████████▉| 1482/1489 [01:06<00:00, 24.75it/s]


0: 640x384 2 persons, 1 sports ball, 10.6ms
Speed: 2.1ms preprocess, 10.6ms inference, 3.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 persons, 1 sports ball, 10.5ms
Speed: 1.9ms preprocess, 10.5ms inference, 5.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 persons, 1 sports ball, 10.1ms
Speed: 3.8ms preprocess, 10.1ms inference, 4.5ms postprocess per image at shape (1, 3, 640, 384)


100%|█████████▉| 1485/1489 [01:06<00:00, 25.17it/s]


0: 640x384 3 persons, 11.6ms
Speed: 1.9ms preprocess, 11.6ms inference, 4.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 persons, 10.0ms
Speed: 1.8ms preprocess, 10.0ms inference, 3.7ms postprocess per image at shape (1, 3, 640, 384)


100%|█████████▉| 1487/1489 [01:07<00:00, 22.19it/s]


In [12]:
import os
print(os.getcwd())

/content/pretrained


And it's a wrap and it's a success.